# Fine-Tuning Laya on Custom Data

**What you'll learn**
- Turn a problem statement into one typed question
- Measure Laya on it before any training
- Fine-tune it with `laya`'s own trainer: one call trains, calibrates and saves
- How many labels you actually need
- Check what the model forgot
- Load the result like the original

## Setup
- `laya.train`: Laya's fine-tuning, `finetune` plus its settings, `TrainConfig`
- `laya.evals`: Laya's evaluation, accuracy and calibration on labelled rows

In [ ]:
import os
import pandas as pd
from datasets import load_dataset
import laya
from laya.train import finetune, TrainConfig
from laya.evals import Dataset, evaluate

In [ ]:
BASE = "convaiinnovations/laya"
agent = laya.Agent(BASE, device="cuda")  # the published Laya
agent.cfg["encoder"]

In [ ]:
agent.model.encoder.num_parameters()

## The Task: Route Bank Card Messages to Ten Intents
- A digital bank's card team gets messages that must reach the right queue
- Ten card intents, several easy to confuse
- Data: Banking77, real customer messages

In [ ]:
INTENTS = {
    "card_arrival": "ordered card has not arrived yet",
    "card_delivery_estimate": "asks how long card delivery takes",
    "activate_my_card": "how to activate a card",
    "card_not_working": "card stopped working or is not accepted",
    "lost_or_stolen_card": "card was lost or stolen",
    "compromised_card": "card details may be known to someone else",
    "card_payment_not_recognised": "a card payment the customer did not make",
    "declined_card_payment": "a card payment was declined",
    "card_swallowed": "an ATM kept the card",
    "contactless_not_working": "tap to pay does not work",
}
len(INTENTS)

In [ ]:
QUESTION = {
    "type": "choice",
    "instructions": "What does the customer need help with?",
    "criteria": INTENTS,  # the ten queues, each with a short description
}
QUESTION

## Build Train and Test Files in Laya's Format
- The same JSONL rows as the previous section, one per message
- `finetune` reads them to train, `laya.evals` reads them to score

In [ ]:
banking = load_dataset("mteb/banking77")  # 77 intents, real customer messages
banking

In [ ]:
def is_card_intent(row):
    return row["label_text"] in INTENTS  # one of our ten card intents?

cards = banking.filter(is_card_intent)
cards

In [ ]:
message = cards["train"][0]  # one Banking77 message
message

In [ ]:
row = {
    "state": message["text"],  # what the customer wrote
    "questions": {"intent": QUESTION},  # the same question for every message
    "expected": {"intent": message["label_text"]},  # the right queue
}
row

In [ ]:
def to_rows(messages):
    rows = []
    for message in messages:
        rows.append({
            "state": message["text"],
            "questions": {"intent": QUESTION},
            "expected": {"intent": message["label_text"]},
        })
    return rows

train_rows = to_rows(cards["train"])
test_rows = to_rows(cards["test"])
len(train_rows), len(test_rows)

In [ ]:
train_table = pd.DataFrame(train_rows)
test_table = pd.DataFrame(test_rows)
train_table.head()

In [ ]:
os.makedirs("runs/cards", exist_ok=True)
train_table.to_json("runs/cards/train.jsonl", orient="records", lines=True)
test_table.to_json("runs/cards/test.jsonl", orient="records", lines=True)
os.listdir("runs/cards")

In [ ]:
with open("runs/cards/train.jsonl") as f:
    first_line = f.readline()  # one row, exactly as finetune reads it
print(first_line)

## Baseline Accuracy Before Fine-Tuning
- `evaluate`: every test message through the agent, scored against `expected`
- Accuracy, expected calibration error (ECE), Brier score

In [ ]:
test = Dataset.from_jsonl("runs/cards/test.jsonl")
len(test)

In [ ]:
before = evaluate(agent, test)  # every test message through the published Laya
before.overall.keys()

In [ ]:
overall = pd.Series(before.overall)
overall[["choice_accuracy", "ece", "brier"]]

In [ ]:
cases = pd.DataFrame(before.cases)  # one row per test message
cases[["expected", "answer", "correct"]].head()

In [ ]:
wrong = cases[~cases["correct"]]  # the messages Laya got wrong
len(wrong)

In [ ]:
mistakes = wrong["expected"] + " → " + wrong["answer"].str.get("choice")  # right queue → Laya's answer
mistakes.head()

In [ ]:
mistakes.value_counts().head(6)

## Fine-Tune Laya with Laya 'finetune'
- One call: hold out 10% of the rows for calibration, train, fit the temperatures, save a checkpoint
- `TrainConfig`: the defaults train the way Laya was trained: every weight, 4 epochs, Laya's `rlcd` loss
- `eval_data`: score the test rows before and after, with the new temperatures

In [ ]:
config = TrainConfig(eval_data="runs/cards/test.jsonl")
config

In [ ]:
OUT = "runs/laya-cards"
run = finetune("runs/cards/train.jsonl", BASE, OUT, config)
run.keys()

In [ ]:
run["train_items"], run["calibration_items"], run["eval_items"]

In [ ]:
run["epoch_loss"]

In [ ]:
run["temperature"]

In [ ]:
scores_before = run["train_report"]["before"]  # the test scores before training
scores_before

In [ ]:
scores_after = run["train_report"]["after"]  # and after
table = pd.DataFrame([scores_before, scores_after], index=["Laya", "Laya fine-tuned"])
table.drop(columns="by_type")

## Before vs After: The Fine-Tuned Laya's Accuracy
- `laya.Agent` takes the output folder the same way it takes a Hub repo

In [ ]:
tuned = laya.Agent(OUT, device="cuda")  # the fine-tuned Laya
tuned.cfg["encoder"]

In [ ]:
after = evaluate(tuned, test)  # every test message through the fine-tuned Laya
after.overall.keys()

In [ ]:
overall_after = pd.Series(after.overall)
overall_after[["choice_accuracy", "ece", "brier"]]

In [ ]:
cases_after = pd.DataFrame(after.cases)
wrong = cases_after[~cases_after["correct"]]  # the messages it still gets wrong
len(wrong)

In [ ]:
mistakes = wrong["expected"] + " → " + wrong["answer"].str.get("choice")
mistakes.value_counts().head(6)

In [ ]:
right_before = cases.groupby("expected")["correct"].mean()  # share right per intent, before
right_before

In [ ]:
right_after = cases_after.groupby("expected")["correct"].mean()  # and after
right_after

In [ ]:
per_intent = pd.DataFrame({
    "before": right_before,
    "after": right_after,
})
per_intent.sort_values("before")

## Forgetting: Did Fine-Tuning Hurt Laya's Other Skills?
- A held-out set from the Kev repo with nothing to do with bank cards: six public datasets plus two groups of Kev's own questions
- MMLU, SciQ, Emotion, offensive tweets, QNLI, PAWS
- Laya reads only type, instructions and criteria: the label can stay in the question, a copy goes to `expected`
- `evaluate` reports every tag separately

In [ ]:
kev = pd.read_json("../vendor/kev/evals/v4/transfer-v4/development.jsonl", lines=True)
kev.head()

In [ ]:
expected = []
tags = []
for record in kev.to_dict("records"):  # one Kev record at a time
    answers = {}
    for qid, question in record["questions"].items():
        answers[qid] = question["label"]  # the gold answer, where Laya looks for it
    expected.append(answers)
    tags.append([record["_meta"]["source"]])  # the task name, so evaluate reports each task

expected[:3], tags[:3]

In [ ]:
kev["expected"] = expected
kev["tags"] = tags
kev[["questions", "expected", "tags"]].head()

In [ ]:
kev.to_json("runs/cards/transfer.jsonl", orient="records", lines=True)
transfer = Dataset.from_jsonl("runs/cards/transfer.jsonl")
len(transfer)

In [ ]:
forget_before = evaluate(agent, transfer)  # the published Laya
by_task_before = pd.DataFrame.from_dict(forget_before.slices["tag"], orient="index")  # one row per task
by_task_before[["choice_accuracy", "noul_accuracy"]]

In [ ]:
forget_after = evaluate(tuned, transfer)  # the fine-tuned Laya
by_task_after = pd.DataFrame.from_dict(forget_after.slices["tag"], orient="index")
by_task_after[["choice_accuracy", "noul_accuracy"]]

In [ ]:
forget = pd.DataFrame({
    "Laya, choice": by_task_before["choice_accuracy"],
    "fine-tuned, choice": by_task_after["choice_accuracy"],
    "Laya, yes or no": by_task_before["noul_accuracy"],
    "fine-tuned, yes or no": by_task_after["noul_accuracy"],
})
forget.round(3)

## Accuracy vs Number of Training Examples
- The same `finetune`, `n` messages per intent, always the same test set
- Each run starts again from the published Laya

In [ ]:
train = pd.read_json("runs/cards/train.jsonl", lines=True)
train.head()

In [ ]:
train["intent"] = train["expected"].str.get("intent")  # the label as its own column
train["intent"].value_counts()

In [ ]:
subset = train.groupby("intent").head(5)  # the first 5 messages of each intent
subset["intent"].value_counts()

In [ ]:
curve = []
for n in [5, 10, 25, 50, 100]:
    subset = train.groupby("intent").head(n)  # the first n messages of each intent
    path = f"runs/cards/train_{n}.jsonl"
    subset.to_json(path, orient="records", lines=True)
    run_n = finetune(path, BASE, f"runs/laya-cards-{n}", config)  # starts again from the published Laya
    accuracy = run_n["train_report"]["after"]["accuracy"]
    curve.append({"per intent": n, "trained on": run_n["train_items"], "accuracy": accuracy})

curve

In [ ]:
pd.DataFrame(curve)

## Lab: Compare the Original and Fine-Tuned Laya on New Messages
- Original and fine-tuned on new messages

In [ ]:
MESSAGES = [
    "Someone might have seen my card number, what should I do?",
    "I got a text saying my card is on its way but it's been two weeks.",
    "The cash machine didn't give my card back!",
    "Why won't my card work when I tap it at the shop?",
]

In [ ]:
old = agent.predict(MESSAGES[0], {"intent": QUESTION})  # the published Laya on one message
old["answers"]["intent"]

In [ ]:
results = []
for message in MESSAGES:
    old = agent.predict(message, {"intent": QUESTION})  # the published Laya
    new = tuned.predict(message, {"intent": QUESTION})  # the fine-tuned Laya
    old_answer = old["answers"]["intent"]
    new_answer = new["answers"]["intent"]
    results.append({
        "message": message,
        "Laya": old_answer["choice"],
        "Laya confidence": round(old_answer["answer_confidence"], 2),
        "fine-tuned": new_answer["choice"],
        "fine-tuned confidence": round(new_answer["answer_confidence"], 2),
    })

results[0]

In [ ]:
pd.DataFrame(results)

## Cheat Sheet
| Step | Code |
|---|---|
| Problem | one `choice` question, criteria describe each queue |
| Data | JSONL rows: `state`, `questions`, `expected` |
| Score | `evaluate(agent, Dataset.from_jsonl(path))`: accuracy, ECE, Brier |
| Fine-tune | `finetune(data, base, out, TrainConfig(eval_data=test))` |
| Calibration | done inside `finetune`, on its own 10% hold-out |
| Forgetting | `tags` per task, `report.slices["tag"]` |
| Load | `laya.Agent(out)`, same API as the published model |